In [1]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
print(PROJECT_ROOT)

/Users/florianb/Downloads/ai-customer-insights-engine


In [2]:
from langchain_huggingface import HuggingFaceEmbeddings

from src.rag import retriever as retriever_module

from config import config

In [3]:
import importlib

importlib.reload(config)
importlib.reload(retriever_module)

<module 'src.rag.retriever' from '/Users/florianb/Downloads/ai-customer-insights-engine/src/rag/retriever.py'>

In [3]:
print(f"HUGGINGFACE_EMBEDDING_MODEL = {config.HUGGINGFACE_EMBEDDING_MODEL}")
print(f"CHROMA_PATH = {config.CHROMA_PATH}")
print(f"COLLECTION_NAME = {config.COLLECTION_NAME}")
print(f"SEARCH_TYPE = {config.SEARCH_TYPE}")
print(f"RETRIEVER_K = {config.RETRIEVER_K}")
print(f"HUGGINGFACE_CROSSENCODER_MODEL = {config.HUGGINGFACE_CROSSENCODER_MODEL}")
print(f"RERANKER_TOP_N = {config.RERANKER_TOP_N}")

HUGGINGFACE_EMBEDDING_MODEL = sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
CHROMA_PATH = /Users/florianb/Downloads/ai-customer-insights-engine/data/chroma_db
COLLECTION_NAME = bank_customer_reviews
SEARCH_TYPE = similarity
RETRIEVER_K = 20
HUGGINGFACE_CROSSENCODER_MODEL = BAAI/bge-reranker-v2-m3
RERANKER_TOP_N = 5


### Validation de `load_retriever()`

In [4]:
embedding_function = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
)

In [27]:
retriever = retriever_module.load_retriever(
    embedding_function=embedding_function,
    chroma_path=PROJECT_ROOT / "data/chroma_db_test_function",
    collection_name="bank_customer_reviews",
    search_type="similarity",
    retriever_k=20,
    reranker_model="BAAI/bge-reranker-v2-m3",
    reranker_top_n=5,
)

In [28]:
type(retriever)

langchain_classic.retrievers.contextual_compression.ContextualCompressionRetriever

In [29]:
retriever.invoke("Quels sont les problèmes avec le service client ?")

[Document(id='8b041ddb-38e1-4312-9daa-655e8e25df61', metadata={'experience_date': '2024-07-13T00:00:00', 'bank': 'boursobank', 'rating': 1, 'review_id': 2551, 'title': 'Le service client...', 'dataset': 'processed_reviews_test_notebook', 'publication_date': '2024-07-13T12:25:42+02:00'}, page_content="La gestion des comptes, l'application, sur se point rien à dire. Mais le service client... Pour de simples demandes il faut parfois envoyer plusieurs mail car il n'y a pas de suivi de discussion d'un mail à l'autre et ils répondent très souvent à coté de la plaque. J'écris cette avis par connaissance de cause, je suis client depuis octobre et aujourd'hui (juillet) il y a toujours des demandes concernant des transfères qui ne sont pas fini car si on ne peux pas passer par leur système automatique le service client ne veut rien entendre et nous renvoi systématiquement sur se système."),
 Document(id='87446aa2-e234-4162-acd2-728a2f90d1e0', metadata={'bank': 'boursobank', 'publication_date': '

### Validation du pipeline `create_retriever()` via le notebook

In [4]:
retriever_notebook = retriever_module.create_retriever(
    model_name=config.HUGGINGFACE_EMBEDDING_MODEL,
    use_reranker=True,
)

In [5]:
type(retriever_notebook)

langchain_classic.retrievers.contextual_compression.ContextualCompressionRetriever

In [6]:
retriever_notebook.invoke("Quels sont les problèmes avec le service client ?")

[Document(id='867d6431-52d8-465f-9dae-9e5887fc8349', metadata={'review_id': 30215, 'publication_date': '2024-11-06T10:22:44+01:00', 'dataset': 'processed_reviews', 'title': 'Deçu', 'rating': 3, 'experience_date': '2024-07-04T00:00:00', 'bank': 'hellobank'}, page_content='Service client pas au top.'),
 Document(id='1ccab572-946b-4b25-a9af-b345c469f2fc', metadata={'rating': 1, 'bank': 'fortuneo', 'publication_date': '2023-03-16T16:48:47+01:00', 'experience_date': '2023-03-08T00:00:00', 'review_id': 17753, 'dataset': 'processed_reviews', 'title': 'Client abandonné'}, page_content="Le service clientele est une catastrophe en terme de qualité et d'information fournie erronée, et ne répond jamais aux réclamations, demandes d'information.... Merci pour vos excuses, je les accepte. En revanche les conséquences de cette erreur sont lourdes (360€/an) sans médiation, compensation ou geste commercial. Il est dommage d'obtenir des réponses en 24h sur une plate-forme d'avis client et de rester sans 